# LangGraph 002 — Generative AI vs Agentic AI

The hiring assistant in four stages. First we count who does the work; then we
build a tiny stage-3 assistant and a tiny stage-4 agent in plain Python and see
which one notices the problem by itself. **No API key, no model** — the job site
is simulated so every run is the same.

| Part | What we check |
|---|---|
| A | by hand: 6, 5, 2, 0 of ten actions across the four stages |
| B | a tool-using assistant never notices the low applications unless asked |
| C | an agent loop notices on its own, asks for approval, and adapts |

## Part A — Who does each step

In [ ]:
# The four stages of the hiring assistant, as data. For each step of the job:
# "hand"   - the recruiter does it, the system does not help
# "draft"  - the system writes it, the recruiter carries it out
# "system" - the system does it (the recruiter may approve)
STEPS = ["draft the JD", "post the JD", "notice few applications", "change the plan",
         "screen resumes", "schedule interviews", "prepare questions",
         "send the offer", "track the reply", "start onboarding"]

STAGES = {
    "1 plain LLM chatbot": ["draft", "hand", "hand", "hand", "hand",
                            "draft", "draft", "draft", "hand", "hand"],
    "2 RAG chatbot":       ["draft", "hand", "hand", "hand", "draft",
                            "draft", "draft", "draft", "hand", "hand"],
    "3 chatbot + tools":   ["draft", "system", "hand", "system", "system",
                            "system", "draft", "system", "hand", "system"],
    "4 agent":             ["system"] * 10,
}

for name, who in STAGES.items():
    print(f"{name:<20} by hand {who.count('hand'):>2}   drafted {who.count('draft'):>2}"
          f"   done by the system {who.count('system'):>2}")

In [ ]:
hand = [who.count("hand") for who in STAGES.values()]
assert hand == [6, 5, 2, 0]
for step, a, b in zip(STEPS, STAGES["3 chatbot + tools"], STAGES["4 agent"]):
    if a != b:
        print(f"  {step:<24} stage 3: {a:<7} stage 4: {b}")

This is a reading of the scenario, not a measurement. If you read a cell
differently, change it and the counts follow.

## Part B — A stage-3 assistant: tools, but only when asked

A simulated job site. Applications arrive slowly unless the role is widened.

In [ ]:
class JobSite:
    def __init__(self):
        self.day, self.total, self.widened = 0, 0, False
    def next_day(self):
        self.day += 1
        self.total += 3 if self.widened else (1 if self.day % 2 == 0 else 0)
    def applications(self):           # a tool
        return self.total
    def widen_role(self):             # a tool
        self.widened = True

def tool_assistant(site, request):
    """Does what it is asked, using tools. Nothing else."""
    if request == "how many applications?":
        return f"{site.applications()} so far"
    return "ok"

site = JobSite()
log = []
for _ in range(7):
    site.next_day()
    # the recruiter is busy and never asks
print(f"after {site.day} days: {site.applications()} applications; "
      f"widened: {site.widened}")
assert site.applications() == 3 and not site.widened

The tools worked. The assistant could have checked at any time. It never did,
because nobody asked.

## Part C — A stage-4 agent: a goal, a loop, and a check every day

In [ ]:
def agent(site, goal=8, days=7, expected_per_day=1, approve=lambda msg: True):
    events = []
    for _ in range(days):
        site.next_day()
        have = site.applications()                     # it looks, unasked
        if have >= goal:
            events.append((site.day, f"goal reached: {have}"))
            break
        behind = have < expected_per_day * site.day
        if behind and not site.widened and site.day >= 3:
            msg = f"day {site.day}: only {have} applications. Widen the role?"
            if approve(msg):                           # a person decides
                site.widen_role()
                events.append((site.day, "widened the role, with approval"))
    return events

site = JobSite()
for day, event in agent(site):
    print(f"day {day}: {event}")
print(f"total after day {site.day}: {site.applications()}")

In [ ]:
site = JobSite()
events = agent(site)
assert events[0] == (3, "widened the role, with approval")
assert events[-1][1].startswith("goal reached") and site.applications() >= 8

# Without approval, the agent notices but does not act:
site = JobSite()
assert agent(site, approve=lambda msg: False) == []
print("noticed by itself; acted only with approval")

Same tools, same job site. The difference is the **loop around the goal**: the
agent looks every day, compares with what it expected, and proposes a change —
then waits for a person to approve it. That loop, with a state that remembers
where the job stands, is what LangGraph is built to express.

## What to take away

- Generative AI is **reactive**; tools alone do not change that.
- An agent **takes the first step**, **remembers** where it is, and **adapts**.
- Approval is part of the design: the agent above changes nothing without it.

## Exercises

1. Change `expected_per_day` to 2. On which day does the agent act now?
2. Give the agent a `track_reply` step: after an offer, check each day until
   the candidate answers.
3. What should the agent do if applications are still low two days after
   widening? Add that rule.